In [1]:
import pandas as pd
import numpy as np
import logging
from Bio import SeqIO
import re

logging.basicConfig(level=logging.WARNING, format="%(message)s")

In [2]:
# preview all the datasets (csv) in this folder

antigen = pd.read_csv("antigen_table_export_1772488096.csv")
epitope = pd.read_csv("epitope_table_export_1772488012.csv")
receptor = pd.read_csv("receptor_table_export_1772488135.csv")
reference_table = pd.read_csv("reference_table_export_1772488149.csv")
tcell = pd.read_csv("tcell_table_export_1772488543.csv")
mhc = pd.read_csv("mhc_ligand_table_export_1772488551.csv")

/var/folders/y_/n7t8whz54_q5s7xfy2s01ypw0000gn/T/ipykernel_44726/1690475545.py:4: DtypeWarning: Columns (9,10,11,12,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  epitope = pd.read_csv("epitope_table_export_1772488012.csv")
/var/folders/y_/n7t8whz54_q5s7xfy2s01ypw0000gn/T/ipykernel_44726/1690475545.py:7: DtypeWarning: Columns (13,14,19,20,21,22,23,24,25,26,46,47,48,49,50,55,56,57,58,61,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,106,107,108,109,110,111,112,113,114,116,121,124,129,133,135,136,142,143,144,145,146,150,153,154,155,156,157,158,159,160,161) have mixed types. Specify dtype option on import or set low_memory=False.
  tcell = pd.read_csv("tcell_table_export_1772488543.csv")


In [3]:
# MHC Columns
mhc_columns = ['Epitope - Epitope IRI',
               'MHC Restriction - Name',
               'MHC Restriction - Class',
               'Epitope - Name',
               'Epitope - Mutation',
               'Assay - Method',
               'Assay - Response measured',
               'Assay - Qualitative Measurement',
               'Assay - Units',
               'Assay - Quantitative measurement']

# keep only mhc_columns in mhc
mhc = mhc[mhc_columns]

# rename columns in mhc
mhc = mhc.rename(columns={
    'Assay - Method': 'MHC Assay - Method',
    'Assay - Response measured': 'MHC Assay - Response measured',
    'Assay - Qualitative Measurement': 'MHC Assay - Qualitative Measurement',
    'Assay - Units': 'MHC Assay - Units',
    'Assay - Quantitative measurement': 'MHC Assay - Quantitative measurement' })

In [4]:
# join on tcell, keep all rows in both df, keep column only once, rename columns in tcell to have prefix "T cell Assay - "
tcell_columns = ['Epitope - CEDAR IRI',
                 'Assay - Method',
                 'Assay - Response measured',
                 'Assay - Qualitative Measurement']

# keep only tcell_columns in tcell
tcell = tcell[tcell_columns]

# rename tcell columns
tcell = tcell.rename(columns={
    'Assay - Method': 'T cell Assay - Method',
    'Assay - Response measured': 'T cell Assay - Response measured',
    'Assay - Qualitative Measurement': 'T cell Assay - Qualitative Measurement' })

# merge mhc and tcell on epitope id, keep all rows in both df, keep column only once
combined = pd.merge(mhc, tcell, left_on='Epitope - Epitope IRI', right_on='Epitope - CEDAR IRI', how='outer')

# remove 'Epitope - CEDAR IRI' column
combined = combined.drop(columns=['Epitope - CEDAR IRI'])

print(combined.shape)
combined.head()

(48906, 13)


,Epitope - Epitope IRI,MHC Restriction - Name,MHC Restriction - Class,Epitope - Name,Epitope - Mutation,MHC Assay - Method,MHC Assay - Response measured,MHC Assay - Qualitative Measurement,MHC Assay - Units,MHC Assay - Quantitative measurement,T cell Assay - Method,T cell Assay - Response measured,T cell Assay - Qualitative Measurement
0,https://cedar.iedb.org/epitope/1025113,HLA-B*13:02,I,KLPKYGVKV,E82K,purified MHC/direct/fluorescence,dissociation constant KD (~EC50),Positive-Low,nM,1000.0,biological activity,activation,Positive
1,https://cedar.iedb.org/epitope/1025113,HLA-B*13:02,I,KLPKYGVKV,E82K,purified MHC/direct/fluorescence,dissociation constant KD (~EC50),Positive-Low,nM,1000.0,biological activity,activation,Positive-Low
2,https://cedar.iedb.org/epitope/1025113,HLA-A2,I,KLPKYGVKV,E82K,mass spectrometry,ligand presentation,Positive,NaN,NaN,biological activity,activation,Positive
3,https://cedar.iedb.org/epitope/1025113,HLA-A2,I,KLPKYGVKV,E82K,mass spectrometry,ligand presentation,Positive,NaN,NaN,biological activity,activation,Positive-Low
4,https://cedar.iedb.org/epitope/1063259,HLA class I,I,DANSFLQSV,P747S,cellular MHC/mass spectrometry,ligand presentation,Positive,NaN,NaN,ELISA,IFNg release,Positive


In [5]:
receptor.columns

Index(['Receptor - Group IRI', 'Receptor - CEDAR Receptor ID',
       'Receptor - Reference Name', 'Receptor - Type', 'Reference - CEDAR IRI',
       'Epitope - CEDAR IRI', 'Epitope - Name', 'Epitope - Source Molecule',
       'Epitope - Source Organism', 'Assay - Type', 'Assay - CEDAR IDs',
       'Assay - MHC Allele Names', 'Chain 1 - Type', 'Chain 1 - Organism IRI',
       'Chain 1 - Nucleotide Sequence', 'Chain 1 - Curated V Gene',
       'Chain 1 - Calculated V Gene', 'Chain 1 - Curated D Gene',
       'Chain 1 - Calculated D Gene', 'Chain 1 - Curated J Gene',
       'Chain 1 - Calculated J Gene', 'Chain 1 - Protein Sequence',
       'Chain 1 - Protein IRI', 'Chain 1 - CDR3 Curated',
       'Chain 1 - CDR3 Calculated', 'Chain 1 - CDR3 Start Curated',
       'Chain 1 - CDR3 End Curated', 'Chain 1 - CDR3 Start Calculated',
       'Chain 1 - CDR3 End Calculated', 'Chain 1 - CDR1 Curated',
       'Chain 1 - CDR1 Calculated', 'Chain 1 - CDR1 Start Curated',
       'Chain 1 - CDR1 End C

In [6]:
# receptor
receptor_columns = ['Epitope - CEDAR IRI',
                    'Chain 2 - CDR3 Curated']

# keep only receptor_columns in receptor
receptor = receptor[receptor_columns]

# keep only receptor_columns in receptor
receptor = receptor[receptor_columns]
# rename receptor columns
receptor = receptor.rename(columns={
    'Chain 2 - CDR3 Curated': 'CDR3β Sequence'})

# merge combined and receptor on epitope id, keep all rows in both df, keep column only once
final = pd.merge(combined, receptor, left_on='Epitope - Epitope IRI', right_on='Epitope - CEDAR IRI', how='outer')
# remove 'Epitope - CEDAR IRI' column
final = final.drop(columns=['Epitope - CEDAR IRI'])
print(final.shape)
final.head()

(51429, 14)


,Epitope - Epitope IRI,MHC Restriction - Name,MHC Restriction - Class,Epitope - Name,Epitope - Mutation,MHC Assay - Method,MHC Assay - Response measured,MHC Assay - Qualitative Measurement,MHC Assay - Units,MHC Assay - Quantitative measurement,T cell Assay - Method,T cell Assay - Response measured,T cell Assay - Qualitative Measurement,CDR3β Sequence
0,https://cedar.iedb.org/epitope/1025113,HLA-B*13:02,I,KLPKYGVKV,E82K,purified MHC/direct/fluorescence,dissociation constant KD (~EC50),Positive-Low,nM,1000.0,biological activity,activation,Positive,NaN
1,https://cedar.iedb.org/epitope/1025113,HLA-B*13:02,I,KLPKYGVKV,E82K,purified MHC/direct/fluorescence,dissociation constant KD (~EC50),Positive-Low,nM,1000.0,biological activity,activation,Positive-Low,NaN
2,https://cedar.iedb.org/epitope/1025113,HLA-A2,I,KLPKYGVKV,E82K,mass spectrometry,ligand presentation,Positive,NaN,NaN,biological activity,activation,Positive,NaN
3,https://cedar.iedb.org/epitope/1025113,HLA-A2,I,KLPKYGVKV,E82K,mass spectrometry,ligand presentation,Positive,NaN,NaN,biological activity,activation,Positive-Low,NaN
4,https://cedar.iedb.org/epitope/1063259,HLA class I,I,DANSFLQSV,P747S,cellular MHC/mass spectrometry,ligand presentation,Positive,NaN,NaN,ELISA,IFNg release,Positive,NaN


In [7]:
# qualitative training df
qual_columns = ['MHC Restriction - Name',
               'MHC Restriction - Class',
               'Epitope - Name',
               'CDR3β Sequence',
               'MHC Assay - Qualitative Measurement',
               'T cell Assay - Qualitative Measurement']

qual_df = final[qual_columns]

# Filter for only MHC Restriction Class - I
qual_df = qual_df[qual_df['MHC Restriction - Class'] == 'I']

# print all possible values for MHC Assay - Qualitative Measurement and T cell Assay - Qualitative Measurement
print(qual_df['MHC Assay - Qualitative Measurement'].value_counts())
print(qual_df['T cell Assay - Qualitative Measurement'].value_counts())

# convert qualitative measurements to labels from 0 to 1
# Positive                 9409
# Negative                 2334
# Positive-Low             1406
# Positive-High            1040
# Positive-Intermediate     447

def convert_qualitative_measurement(x):
    if x == 'Positive':
        return 1
    elif x == 'Negative':
        return 0
    elif x == 'Positive-Low':
        return 0.25
    elif x == 'Positive-High':
        return 0.75
    elif x == 'Positive-Intermediate':
        return 0.5
    else:
        return np.nan

qual_df['MHC Assay - Qualitative Measurement'] = qual_df['MHC Assay - Qualitative Measurement'].apply(convert_qualitative_measurement)
qual_df['T cell Assay - Qualitative Measurement'] = qual_df['T cell Assay - Qualitative Measurement'].apply(convert_qualitative_measurement)

# remove all MHC Restriction - Name that do not have HLA-A, B, or C with *
qual_df = qual_df[qual_df['MHC Restriction - Name'].str.contains(r'HLA-[ABC]\*\d+', regex=True)]

# remove column MHC Restriction - Class
qual_df = qual_df.drop(columns=['MHC Restriction - Class'])

# drop row if 'Epitope - Name' has '+'
qual_df = qual_df[~qual_df['Epitope - Name'].str.contains(r'\+', regex=True)]

print(qual_df.shape)
qual_df.head()

MHC Assay - Qualitative Measurement
Positive                 9409
Negative                 2334
Positive-Low             1406
Positive-High            1040
Positive-Intermediate     447
Name: count, dtype: int64
T cell Assay - Qualitative Measurement
Positive                 8654
Negative                 3301
Positive-Low             1319
Positive-High             189
Positive-Intermediate      85
Name: count, dtype: int64
(14349, 5)


,MHC Restriction - Name,Epitope - Name,CDR3β Sequence,MHC Assay - Qualitative Measurement,T cell Assay - Qualitative Measurement
0,HLA-B*13:02,KLPKYGVKV,NaN,0.25,1.00
1,HLA-B*13:02,KLPKYGVKV,NaN,0.25,0.25
6,HLA-B*07:02,APAWSPTHPL,NaN,1.00,NaN
7,HLA-B*08:01,DGTRSITL,NaN,0.00,NaN
8,HLA-B*08:01,EPSFRGITF,NaN,0.00,NaN


In [8]:
# rename columns
qual_df = qual_df.rename(columns={
    'MHC Restriction - Name': 'mhc_allele',
    'Epitope - Name': 'epitope',
    'CDR3β Sequence': 'tcell_sequence',
    'MHC Assay - Qualitative Measurement': 'mhc_binding',
    'T cell Assay - Qualitative Measurement': 'tcell_binding'})

qual_df.head()

,mhc_allele,epitope,tcell_sequence,mhc_binding,tcell_binding
0,HLA-B*13:02,KLPKYGVKV,NaN,0.25,1.00
1,HLA-B*13:02,KLPKYGVKV,NaN,0.25,0.25
6,HLA-B*07:02,APAWSPTHPL,NaN,1.00,NaN
7,HLA-B*08:01,DGTRSITL,NaN,0.00,NaN
8,HLA-B*08:01,EPSFRGITF,NaN,0.00,NaN


In [9]:
hla_fasta = "/Users/vivianschu/repos/AETHER/data/HLA/hla_prot.fasta"

# Regex for 2-field allele (e.g. B*13:02)
FASTA_RE = re.compile(r"\b([A-Z])\*(\d{2}):(\d{2})")
DF_RE = re.compile(r"(?:HLA-)?([A-Z])\*(\d{2}):?(\d{2})")

# Build lookup dictionary: "B*13:02" -> sequence
hla = {}
for rec in SeqIO.parse(hla_fasta, "fasta"):
    m = FASTA_RE.search(rec.description)
    if m:
        key = f"{m.group(1)}*{m.group(2)}:{m.group(3)}"
        hla.setdefault(key, str(rec.seq))  # keep first occurrence


def get_seq(df_allele):
    s = str(df_allele).strip().replace(" ", "")
    m = DF_RE.search(s)

    if not m:
        logging.warning(f"BAD FORMAT: {df_allele}")
        return None

    key = f"{m.group(1)}*{m.group(2)}:{m.group(3)}"
    seq = hla.get(key)

    if seq is None:
        logging.warning(f"MISS: {df_allele} -> {key}")

    return seq


qual_df["mhc_sequence"] = qual_df["mhc_allele"].map(get_seq)

qual_df[["mhc_allele", "mhc_sequence"]].head()

,mhc_allele,mhc_sequence
0,HLA-B*13:02,MRVTAPRTLLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...
1,HLA-B*13:02,MRVTAPRTLLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...
6,HLA-B*07:02,MLVMAPRTVLLLLSAALALTETWAGSHSMRYFYTSVSRPGRGEPRF...
7,HLA-B*08:01,MLVMAPRTVLLLLSAALALTETWAGSHSMRYFDTAMSRPGRGEPRF...
8,HLA-B*08:01,MLVMAPRTVLLLLSAALALTETWAGSHSMRYFDTAMSRPGRGEPRF...


In [10]:
# filter qual_df for non-null rows for:
# df 1: mhc allele, epitope, and mhc qualitative measurement
# df 2: mhc allele, epitope, t cell qualitative measurement
# df 3: mhc allele, epitope, mhc assay qualitative measurement and  t cell assay qualitative measurement
# df 4: mhc allele, epitope, cdr3b sequence, mhc assay qualitative measurement
# df 5: mhc allele, epitope, cdr3b sequence, t cell assay qualitative measurement
# df 6: mhc allele, epitope, cdr3b sequence, mhc assay qualitative measurement, t cell assay qualitative measurement

df1 = qual_df[qual_df['mhc_binding'].notnull()]
df2 = qual_df[qual_df['tcell_binding'].notnull()]
print(df1.shape)
print(df2.shape)

(14349, 6)
(13381, 6)


In [11]:
# df1 drop t cell assay qualitative measurement and cdr3b sequence columns
# df2 drop cdr3b sequence columns
df1 = df1.drop(columns=['tcell_binding', 'tcell_sequence'])
# df1 drop '
df2 = df2.drop(columns=['tcell_sequence'])

# save df1 as mhc_set
# save df2 as mhc_tcell_set
mhc_set = df1
mhc_tcell_set = df2

mhc_set.head()
mhc_tcell_set.head()

,mhc_allele,epitope,mhc_binding,tcell_binding,mhc_sequence
0,HLA-B*13:02,KLPKYGVKV,0.25,1.00,MRVTAPRTLLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...
1,HLA-B*13:02,KLPKYGVKV,0.25,0.25,MRVTAPRTLLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...
25,HLA-A*24:02,LIMQLMPFGCL,0.00,1.00,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFSTSVSRPGRGEPRF...
26,HLA-A*24:02,LIMQLMPFGCL,0.00,0.25,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFSTSVSRPGRGEPRF...
27,HLA-A*24:02,LIMQLMPFGCL,0.00,1.00,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFSTSVSRPGRGEPRF...


In [12]:
# export
mhc_set.to_csv("/Users/vivianschu/repos/AETHER/data/model/mhc_set.csv", index=False)
mhc_tcell_set.to_csv("/Users/vivianschu/repos/AETHER/data/model/mhc_tcell_set.csv", index=False)